# Interactive session with Spark to explore Bronze results

This notebook uses the project virtual environment and repo-local paths only.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd

os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 8g --executor-memory 8g pyspark-shell"

repo_root = Path.cwd().resolve()
if not (repo_root / "bronze").exists():
    repo_root = Path("/home/dcamacho/dev/ProjectData").resolve()

source_sample_dir = repo_root / "sample_data"
source_dir = repo_root / "data/exports/projectA"
table_path = repo_root / "_tmp" / "bronze_pid_documents"

# Remove stale Spark/Python overrides from previous runs
for key in ["PYTHONPATH", "SPARK_HOME", "PYSPARK_PYTHON", "PYSPARK_DRIVER_PYTHON"]:
    os.environ.pop(key, None)
    
sys.path = [p for p in sys.path if "/opt/spark" not in p]

print("repo_root =", repo_root)
print("source_sample_dir =", source_sample_dir)
print("source_dir =", source_dir)
print("table_path =", table_path)
print("python_executable =", sys.executable)

print("Python:", sys.executable)
print("PYTHONPATH:", os.environ.get("PYTHONPATH"))

repo_root = /home/dcamacho/dev/ProjectData
source_sample_dir = /home/dcamacho/dev/ProjectData/sample_data
source_dir = /home/dcamacho/dev/ProjectData/data/exports/projectA
table_path = /home/dcamacho/dev/ProjectData/_tmp/bronze_pid_documents
python_executable = /home/dcamacho/dev/ProjectData/.venv/bin/python
Python: /home/dcamacho/dev/ProjectData/.venv/bin/python
PYTHONPATH: None


Empecemos asegurando que la carpeta donde probaremos el almacenamiento de la tabla sin metastore

In [2]:
import shutil

# Aseguramos que la ruta sea un objeto Path
path_a_eliminar = table_path

if path_a_eliminar.exists() and path_a_eliminar.is_dir():
    # Borra la carpeta y todo lo que tiene adentro de forma recursiva
    shutil.rmtree(path_a_eliminar)
    print(f"La carpeta {path_a_eliminar.name} fue eliminada con éxito.")
else:
    print("La carpeta no existe o ya había sido eliminada.")

La carpeta bronze_pid_documents fue eliminada con éxito.


Verifiquemos la version de spark que estamos correiendo (debe ser la del virtual environment)

In [3]:
import pyspark
print(pyspark.__file__)
print(pyspark.__version__)

/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/__init__.py
3.5.1


Usamos el comando cli para la ingesta de datos en bronze, tomando los datos de muestra (sample), y guardando los resultados en archivos que representan la tabla (sin metastore). 

In [4]:
python_executable = globals().get("python_executable", globals().get("current_python", sys.executable))
source_sample_dir_str = str(source_sample_dir)
source_dir_str = str(source_dir)
table_path_str = str(table_path)

!{python_executable} -m bronze.cli ingest \
    --source-dir "{source_sample_dir_str}" \
    --table-path "{table_path_str}" \
    --project-code Sample

26/08/30 21:59:13 WARN Utils: Your hostname, DC01NNCOL resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/30 21:59:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/dcamacho/.ivy2/cache
The jars for the packages stored in: /home/dcamacho/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4b1cd5d8-0a6b-4ccb-90f4-cc9a6c1a9211;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 205ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-sto

In [5]:
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"


Inicio de sesión en Spark en local para uso didactico, usaremos delta pipeline adicionalmente, y le asignamos 8GB de RAM al driver y executors

In [6]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("Interactive_Bronze_Exploration")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("Spark ready:", spark.version)

your 131072x1 screen size is bogus. expect trouble


:: loading settings :: url = jar:file:/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/dcamacho/.ivy2/cache
The jars for the packages stored in: /home/dcamacho/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4392e7ee-527f-433a-accd-cf3b6bff42c2;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 115ms :: artifacts dl 6ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   | 

Spark ready: 3.5.1


Verifiquemos como quedaron escritos los datos en nuestra tabla

In [7]:
# Load the table created by the CLI ingest run
path = str(table_path)
df = spark.read.format("delta").load(path)
# print("Rows:", df.count())
df.show(50,truncate=100)

26/08/30 21:59:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------------------------+----------------------------------------------------------------------------------------------------+------------+-----------------------------------------------------------------------+---------------+---------------------------------------------------------------------------+---------------------------+--------------------+--------------------------+------------------------------------+------------------+-------------+-----------------------+-----------------------------+-----------------------------+----------------+---------------------+------------+-------------------+---------------+-----------+
|                           bronze_id|                                                                                             content|content_text|                                                           content_hash|file_size_bytes|                                                                source_path|            source_filename|source_la

In [8]:
# Show a compact preview
visible_columns = [
    "document_number",
    "drawing_revision",
    "project_code",
    "source_filename",
    "file_size_bytes",
    "ingested_at",
    "content_text",
]

preview = df.select(*[c for c in visible_columns if c in df.columns]).limit(10).toPandas()
preview

,document_number,drawing_revision,project_code,source_filename,file_size_bytes,ingested_at,content_text
0,216097C-A22-PID-0021-0012-001,G,Sample,projectB_postproc_0012.xml,1684,2026-08-30 21:59:19.006924,None
1,215777C-36292-PID-0031-02231,C,Sample,projectA_dexpi_02231.xml,1515,2026-08-30 21:59:19.006924,None
2,A14-0001-001,A,Sample,projectB_postproc_0001.xml,870,2026-08-30 21:59:19.006924,None
3,215777C-36209-PID-0021-01010,01,Sample,projectA_dexpi_01010.xml,1040,2026-08-30 21:59:19.006924,None
4,None,None,Sample,malformed_truncated.xml,429,2026-08-30 21:59:19.006924,None
5,A22-0007-003,B,Sample,ambiguous_no_originator.xml,495,2026-08-30 21:59:19.006924,None


## Delta history

Las tablas delta guardan su propia historia, luego de cargar la primera vez vemos que solo tiene una versión.

In [9]:
from delta.tables import DeltaTable

try:
    delta_table = DeltaTable.forPath(spark, str(table_path))
    history_df = delta_table.history()
    history_df.select("version", "timestamp", "operation", "operationParameters").show(truncate=False)
except Exception as e:
    print(f"Unable to read Delta history: {e}")

+-------+-----------------------+---------+-------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                    |
+-------+-----------------------+---------+-------------------------------------------------------+
|0      |2026-08-30 21:59:28.387|WRITE    |{mode -> ErrorIfExists, partitionBy -> ["ingest_date"]}|
+-------+-----------------------+---------+-------------------------------------------------------+



Ahora cargamos los datos del proyecto A.

In [10]:
python_executable = globals().get("python_executable", globals().get("current_python", sys.executable))
source_sample_dir_str = str(source_sample_dir)
source_dir_str = str(source_dir)
table_path_str = str(table_path)

!{python_executable} -m bronze.cli ingest \
    --source-dir "{source_dir_str}" \
    --table-path "{table_path_str}" \
    --project-code A

:: loading settings :: url = jar:file:/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/dcamacho/.ivy2/cache
The jars for the packages stored in: /home/dcamacho/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bf97604c-5e83-4616-8c77-a57cf831c064;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 116ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts 

Si volvemos a verificar nuestra tabla, vemos los nuevos archivos del proyecto A, junto a los archivos de test (Sample)

In [11]:
preview = df.select(*[c for c in visible_columns if c in df.columns]).limit(10).toPandas()
preview

,document_number,drawing_revision,project_code,source_filename,file_size_bytes,ingested_at,content_text
0,362-09-PR-PID-01050,01,A,362-09-01050_Dexpi.xml,11926337,2026-08-30 21:59:57.201089,None
1,362-92-PR-PID-02265,01,A,362-92-02265_Dexpi.xml,6669430,2026-08-30 21:59:57.201089,None
2,362-09-PR-PID-01470,01,A,362-09-01470_Dexpi.xml,12537819,2026-08-30 21:59:57.201089,None
3,362-09-PR-PID-01010,01,A,362-09-01010_Dexpi.xml,11819694,2026-08-30 21:59:57.201089,None
4,216097C-A22-PID-0021-0012-001,G,Sample,projectB_postproc_0012.xml,1684,2026-08-30 21:59:19.006924,None
5,215777C-36292-PID-0031-02231,C,Sample,projectA_dexpi_02231.xml,1515,2026-08-30 21:59:19.006924,None
6,A14-0001-001,A,Sample,projectB_postproc_0001.xml,870,2026-08-30 21:59:19.006924,None
7,215777C-36209-PID-0021-01010,01,Sample,projectA_dexpi_01010.xml,1040,2026-08-30 21:59:19.006924,None
8,None,None,Sample,malformed_truncated.xml,429,2026-08-30 21:59:19.006924,None
9,A22-0007-003,B,Sample,ambiguous_no_originator.xml,495,2026-08-30 21:59:19.006924,None


In [12]:
try:
    delta_table = DeltaTable.forPath(spark, str(table_path))
    history_df = delta_table.history()
    history_df.select("version", "timestamp", "operation", "operationParameters").show(truncate=False)
except Exception as e:
    print(f"Unable to read Delta history: {e}")

+-------+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                                                                                                                                     |
+-------+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1      |2026-08-30 22:00:00.487|MERGE    |{predicate -> ["(content_hash#1271 = content_hash#31)"], matchedPredicates -> [], notMatchedPredicates -> [{"actionType":"insert"}], notMatchedBySourcePredicates -> []}|
|0      |2026-08-30 21:59:28.387|WRITE    |{mode -> ErrorIfExists, partitionBy -> ["ingest_date"]}                                                  

En una tabla delta, podemos inspeccionar los datos en versiones anteriores, en este caso en la version 0 (versionAsOf) teníamos solo 6 archivos

In [ ]:
# Optional: time-travel example for version 0 if it exists
try:
    version_zero = spark.read.format("delta").option("versionAsOf", 0).load(str(table_path))
    print("Rows in version 0:", version_zero.count())
    version_zero.show(20, truncate=100)
except Exception as e:
    print(f"Version travel is unavailable yet: {e}")
finally:
    # keep the session alive until you explicitly stop it
    pass

Rows in version 0: 6
+------------------------------------+----------------------------------------------------------------------------------------------------+------------+-----------------------------------------------------------------------+---------------+--------------------------------------------------------------------------+--------------------------+--------------------+--------------------------+------------------------------------+------------------+-------------+-----------------------+-----------------------------+-----------------------------+----------------+---------------------+------------+-------------------+---------------+-----------+
|                           bronze_id|                                                                                             content|content_text|                                                           content_hash|file_size_bytes|                                                               source_path|           source_f

Podemos parar la sesión de spark.

In [14]:
spark.stop()

# Metadata Store

Continuamos ahora probando la ingesta de datos a la capa bronze usando el metastore. En este comando cli se utiliza el hive store activado, lo cual permite salvar los datos dentro del schema bronze, que alberga pid_documents (bronze.pid_documents).

In [15]:
!{python_executable} -m bronze.cli ingest \
    --source-dir {source_dir} \
    --table bronze.pid_documents

:: loading settings :: url = jar:file:/home/dcamacho/dev/ProjectData/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/dcamacho/.ivy2/cache
The jars for the packages stored in: /home/dcamacho/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-903d6caa-9d32-4fbd-a83e-7df7c164f44f;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 110ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts 

Iniciamos una sesión para explorar el resultado. En este caso activando Hive (nuestro metadata store db)

In [22]:
spark = (
    builder
    .appName("MetastoreApp")
    .master("local[*]")
    .enableHiveSupport()  # Required so the metastore_db is persisted
    .getOrCreate()
)

Cuando se usa el metadata store, le damos funcionalidades de una base de datos a spark. Tenemos ahora un namespace llamado bronze, una tabla pid_documents

In [27]:
spark.sql("SHOW TABLES in bronze").show()

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|   bronze|pid_documents|      false|
+---------+-------------+-----------+



Tenemos en el metastore la gestion de los metadatos de la tabla bronze.pid_documents.

In [26]:
spark.sql("DESCRIBE EXTENDED bronze.pid_documents").show(200, truncate=False)

+----------------------------+---------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                  |comment|
+----------------------------+---------------------------------------------------------------------------+-------+
|bronze_id                   |string                                                                     |NULL   |
|content                     |binary                                                                     |NULL   |
|content_text                |string                                                                     |NULL   |
|content_hash                |string                                                                     |NULL   |
|file_size_bytes             |bigint                                                                     |NULL   |
|source_path                 |string                                            

Podemos acceder a la tabla de manera intuitiva.

In [19]:
df = spark.table("bronze.pid_documents")
df.show(50, truncate=100)


+------------------------------------+----------------------------------------------------------------------------------------------------+------------+-----------------------------------------------------------------------+---------------+--------------------------------------------------------------------------------+----------------------+-----------------------+-------------------------+------------------------------------+------------------+-------------+-----------------------+----------------------+-------------------+----------------+---------------------+------------+-------------------+---------------+-----------+
|                           bronze_id|                                                                                             content|content_text|                                                           content_hash|file_size_bytes|                                                                     source_path|       source_filename|   source_last_modified|

In [20]:
# O directamente en SQL:
spark.sql("SELECT * FROM bronze.pid_documents WHERE header_parse_ok = true")


DataFrame[bronze_id: string, content: binary, content_text: string, content_hash: string, file_size_bytes: bigint, source_path: string, source_filename: string, source_last_modified: timestamp, ingested_at: timestamp, ingest_run_id: string, originating_system: string, source_format: string, format_detection_method: string, client_document_number: string, document_number: string, drawing_revision: string, drawing_revision_date: string, project_code: string, project_code_source: string, header_parse_ok: boolean, ingest_date: date]

In [21]:
spark.stop()